# Sentiment140 FULL — RNN / LSTM / GRU

Pełny projekt dydaktyczny do klasyfikacji sentymentu na **Sentiment140 (1,6 mln tweetów)**.

Notebook:
- automatycznie pobiera oryginalny zbiór w Google Colab,
- ma przełącznik `small` (20k), `medium` (200k), `full` (1,6M),
- losuje dane z zachowaniem balansu klas,
- porównuje RNN, LSTM i GRU,
- pokazuje trening, walidację, test, confusion matrix i stress-test,
- zapisuje najlepszy model.

Źródła: Stanford `https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip` oraz kopia Kaggle `https://www.kaggle.com/datasets/kazanova/sentiment140`.

> Oryginalne etykiety Sentiment140: `0 = negative`, `4 = positive`. W notebooku mapujemy `4 -> 1`.

In [ ]:
# 1. Importy i powtarzalność
import os, re, sys, time, random, zipfile, subprocess, urllib.request
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:',torch.__version__)
print('Urządzenie:',DEVICE)
if DEVICE.type=='cpu': print('Dla medium/full w Colab włącz GPU: Runtime -> Change runtime type -> GPU')

## 2. Konfiguracja
Zmieniaj przede wszystkim `DATASET_SIZE`. Przy `full` warto najpierw ustawić `MODELS_TO_RUN = ["GRU"]`, a dopiero potem trenować wszystkie trzy architektury.

In [ ]:
# 2. KONFIGURACJA
DATASET_SIZE='small'       # 'small' | 'medium' | 'full'
MODELS_TO_RUN=['RNN','LSTM','GRU']
SIZE_MAP={'small':20_000,'medium':200_000,'full':None}
assert DATASET_SIZE in SIZE_MAP

MAX_VOCAB=30_000
MAX_LEN=40
EMBED_DIM=128
HIDDEN_DIM=128
DROPOUT=0.30
NUM_LAYERS=1
BIDIRECTIONAL=False
LEARNING_RATE=1e-3

if DATASET_SIZE=='small': BATCH_SIZE,EPOCHS=128,3
elif DATASET_SIZE=='medium': BATCH_SIZE,EPOCHS=(256 if DEVICE.type=='cuda' else 128),3
else: BATCH_SIZE,EPOCHS=(256 if DEVICE.type=='cuda' else 128),2

print('Tryb:',DATASET_SIZE,'rekordy:',SIZE_MAP[DATASET_SIZE] or 'FULL')
print('Modele:',MODELS_TO_RUN,'batch:',BATCH_SIZE,'epochs:',EPOCHS)

## 3. Automatyczne pobieranie danych
Notebook najpierw szuka pełnego CSV lokalnie. Jeśli go nie ma, próbuje oficjalnego archiwum Stanford, a potem publicznej kopii Kaggle przez `kagglehub`.

Zmienna środowiskowa `S140_SMOKE_TEST=1` służy wyłącznie do technicznego testu notebooka bez internetu.

In [ ]:
# 3. Pobranie / znalezienie Sentiment140
STANFORD_URL='https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip'
KAGGLE_DATASET='kazanova/sentiment140'
FULL_FILENAME='training.1600000.processed.noemoticon.csv'
CACHE_DIR=Path('sentiment140_data'); CACHE_DIR.mkdir(exist_ok=True)

def find_full_csv():
    for p in [Path(FULL_FILENAME),CACHE_DIR/FULL_FILENAME,Path('/content')/FULL_FILENAME,Path('/content/sentiment140_data')/FULL_FILENAME]:
        if p.exists() and p.stat().st_size>100_000_000: return p
    return None

def download_stanford():
    z=CACHE_DIR/'trainingandtestdata.zip'
    print('Pobieram Sentiment140 ze Stanford...')
    urllib.request.urlretrieve(STANFORD_URL,z)
    print('Rozpakowuję...')
    with zipfile.ZipFile(z,'r') as f: f.extractall(CACHE_DIR)
    p=CACHE_DIR/FULL_FILENAME
    if not p.exists(): raise FileNotFoundError(FULL_FILENAME)
    return p

def download_kaggle():
    print('Próbuję źródła zapasowego Kaggle...')
    try: import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','kagglehub'])
        import kagglehub
    folder=Path(kagglehub.dataset_download(KAGGLE_DATASET))
    found=list(folder.rglob(FULL_FILENAME))
    if not found: raise FileNotFoundError(FULL_FILENAME)
    return found[0]

def ensure_data():
    p=find_full_csv()
    if p is not None:
        print('Znaleziono pełny zbiór:',p.resolve()); return p,False
    if os.environ.get('S140_SMOKE_TEST')=='1':
        sample=Path('twitter_sentiment_sample.csv')
        if sample.exists():
            print('SMOKE TEST: lokalny mini-zbiór:',sample.resolve()); return sample,True
    errors=[]
    try: return download_stanford(),False
    except Exception as e:
        errors.append('Stanford: '+repr(e)); print('Stanford niedostępny:',e)
    try: return download_kaggle(),False
    except Exception as e: errors.append('Kaggle: '+repr(e))
    raise RuntimeError('Nie udało się pobrać danych. Pobierz ręcznie '+FULL_FILENAME+' z Kaggle i umieść obok notebooka.\n'+'\n'.join(errors))

DATA_PATH,IS_SMOKE_SAMPLE=ensure_data()
print('DATA_PATH:',DATA_PATH)

In [ ]:
# 4. Wczytanie danych
if IS_SMOKE_SAMPLE:
    data=pd.read_csv(DATA_PATH,usecols=['sentiment','text'])
else:
    cols=['sentiment','id','date','query','user','text']
    data=pd.read_csv(DATA_PATH,encoding='latin-1',header=None,names=cols,usecols=['sentiment','text'])
    data['sentiment']=data['sentiment'].replace({4:1})

data=data.dropna(subset=['text','sentiment']).copy()
data['text']=data['text'].astype(str)
data['sentiment']=data['sentiment'].astype(int)
data=data[data.sentiment.isin([0,1])].reset_index(drop=True)
print('Wczytano:',len(data))
print(data.sentiment.value_counts().sort_index())
data.head()

In [ ]:
# 5. Balansowane próbkowanie: NIE bierzemy po prostu pierwszych N wierszy
def balanced_sample(frame,total_n,seed=42):
    if total_n is None or total_n>=len(frame):
        return frame.sample(frac=1,random_state=seed).reset_index(drop=True)
    per=total_n//2
    parts=[]
    for label in [0,1]:
        g=frame[frame.sentiment==label]
        parts.append(g.sample(n=min(per,len(g)),random_state=seed))
    return pd.concat(parts,ignore_index=True).sample(frac=1,random_state=seed).reset_index(drop=True)

target=SIZE_MAP[DATASET_SIZE]
if IS_SMOKE_SAMPLE: target=min(target or len(data),len(data))
data=balanced_sample(data,target,SEED)
print('Eksperyment:',len(data),'rekordów')
print(data.sentiment.value_counts().sort_index())

In [ ]:
# 6. Szybka EDA
data['char_len']=data.text.str.len()
fig,ax=plt.subplots(figsize=(7,4))
data.sentiment.value_counts().sort_index().plot(kind='bar',ax=ax)
ax.set_title('Balans klas'); ax.set_xlabel('0=negative, 1=positive'); ax.set_ylabel('liczba')
plt.show()
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(data.loc[data.sentiment==0,'char_len'],bins=30,alpha=.6,label='negative')
ax.hist(data.loc[data.sentiment==1,'char_len'],bins=30,alpha=.6,label='positive')
ax.set_title('Długość tweetów'); ax.set_xlabel('znaki'); ax.legend(); plt.show()

## 4. Preprocessing
URL i użytkowników zamieniamy na tokeny specjalne. Zachowujemy treść hashtagów, negację, wykrzykniki i proste emotikony.

In [ ]:
# 7. Czyszczenie i tokenizacja
def clean_text(text):
    text=str(text).lower().strip()
    text=re.sub(r'https?://\S+|www\.\S+',' <url> ',text)
    text=re.sub(r'@\w+',' <user> ',text)
    text=re.sub(r'#(\w+)',r' \1 ',text)
    return re.sub(r'\s+',' ',text).strip()
TOKEN_RE=re.compile(r"<url>|<user>|[a-z]+(?:'[a-z]+)?|[:;=8xX][-^']?[()dDpP/\\]+|[!?]+")
def tokenize(text): return TOKEN_RE.findall(clean_text(text))
data['clean_text']=data.text.map(clean_text)
data[['sentiment','text','clean_text']].head()

In [ ]:
# 8. Podział 70/15/15
train_df,temp_df=train_test_split(data[['sentiment','clean_text']],test_size=.30,random_state=SEED,stratify=data.sentiment)
val_df,test_df=train_test_split(temp_df,test_size=.50,random_state=SEED,stratify=temp_df.sentiment)
print('train',len(train_df),'val',len(val_df),'test',len(test_df))

## 5. Słownik
Słownik budujemy wyłącznie na train, żeby uniknąć leakage'u.

In [ ]:
# 9. Vocabulary
t0=time.time(); counter=Counter()
for text in train_df.clean_text: counter.update(tokenize(text))
itos=['<PAD>','<UNK>']+[w for w,_ in counter.most_common(MAX_VOCAB-2)]
stoi={w:i for i,w in enumerate(itos)}
PAD_IDX=stoi['<PAD>']; UNK_IDX=stoi['<UNK>']
print('Vocabulary:',len(itos),'czas:',round(time.time()-t0,1),'s')
print(counter.most_common(20))
def encode(text,max_len=MAX_LEN):
    ids=[stoi.get(tok,UNK_IDX) for tok in tokenize(text)][:max_len]
    length=max(1,len(ids)); ids+= [PAD_IDX]*(max_len-len(ids))
    return ids,length

In [ ]:
# 10. Dataset i DataLoader
class TweetDataset(Dataset):
    def __init__(self,frame):
        self.texts=frame.clean_text.tolist(); self.labels=frame.sentiment.astype(np.float32).to_numpy()
    def __len__(self): return len(self.texts)
    def __getitem__(self,i):
        ids,L=encode(self.texts[i])
        return torch.tensor(ids),torch.tensor(L),torch.tensor(self.labels[i],dtype=torch.float32)
kw=dict(batch_size=BATCH_SIZE,pin_memory=(DEVICE.type=='cuda'),num_workers=(2 if DEVICE.type=='cuda' else 0))
train_loader=DataLoader(TweetDataset(train_df),shuffle=True,**kw)
val_loader=DataLoader(TweetDataset(val_df),shuffle=False,**kw)
test_loader=DataLoader(TweetDataset(test_df),shuffle=False,**kw)
x,lengths,y=next(iter(train_loader)); print(x.shape,lengths.shape,y.shape)

## 6. Model wspólny dla RNN / LSTM / GRU
`Embedding → warstwa rekurencyjna → Dropout → Linear`. `pack_padded_sequence` ignoruje `<PAD>`.

In [ ]:
# 11. Model
class SentimentRNN(nn.Module):
    def __init__(self,vocab_size,rnn_type='LSTM',embed_dim=128,hidden_dim=128,num_layers=1,dropout=.3,bidirectional=False):
        super().__init__(); self.rnn_type=rnn_type.upper(); self.bidirectional=bidirectional
        self.embedding=nn.Embedding(vocab_size,embed_dim,padding_idx=PAD_IDX)
        common=dict(input_size=embed_dim,hidden_size=hidden_dim,num_layers=num_layers,batch_first=True,dropout=(dropout if num_layers>1 else 0),bidirectional=bidirectional)
        if self.rnn_type=='RNN': self.rnn=nn.RNN(nonlinearity='tanh',**common)
        elif self.rnn_type=='LSTM': self.rnn=nn.LSTM(**common)
        elif self.rnn_type=='GRU': self.rnn=nn.GRU(**common)
        else: raise ValueError('RNN/LSTM/GRU')
        self.dropout=nn.Dropout(dropout); self.fc=nn.Linear(hidden_dim*(2 if bidirectional else 1),1)
    def forward(self,x,lengths):
        emb=self.embedding(x)
        packed=pack_padded_sequence(emb,lengths.cpu(),batch_first=True,enforce_sorted=False)
        if self.rnn_type=='LSTM': _,(h,_)=self.rnn(packed)
        else: _,h=self.rnn(packed)
        h=torch.cat((h[-2],h[-1]),1) if self.bidirectional else h[-1]
        return self.fc(self.dropout(h)).squeeze(1)

In [ ]:
# 12. Trening i ewaluacja
def evaluate(model,loader,criterion):
    model.eval(); loss_sum=n=0; preds=[]; labels=[]
    with torch.no_grad():
        for x,L,y in loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            logits=model(x,L); loss=criterion(logits,y); bs=y.size(0)
            loss_sum+=loss.item()*bs; n+=bs
            preds.extend((torch.sigmoid(logits)>=.5).long().cpu().tolist()); labels.extend(y.long().cpu().tolist())
    return loss_sum/n,accuracy_score(labels,preds)

def train_model(model,train_loader,val_loader,epochs=3,lr=1e-3):
    criterion=nn.BCEWithLogitsLoss(); opt=torch.optim.Adam(model.parameters(),lr=lr)
    H={'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[]}; best=None; best_v=float('inf')
    for ep in range(1,epochs+1):
        t=time.time(); model.train(); loss_sum=n=0; preds=[]; labels=[]
        for x,L,y in train_loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True); logits=model(x,L); loss=criterion(logits,y); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); bs=y.size(0)
            loss_sum+=loss.item()*bs; n+=bs
            preds.extend((torch.sigmoid(logits)>=.5).long().detach().cpu().tolist()); labels.extend(y.long().detach().cpu().tolist())
        tl=loss_sum/n; ta=accuracy_score(labels,preds); vl,va=evaluate(model,val_loader,criterion)
        for k,v in [('train_loss',tl),('train_acc',ta),('val_loss',vl),('val_acc',va)]: H[k].append(v)
        if vl<best_v: best_v=vl; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        print(f'Epoch {ep:02d}/{epochs} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f} | {time.time()-t:.1f}s')
    model.load_state_dict(best); model.to(DEVICE); return H

## 7. Porównanie RNN / LSTM / GRU

In [ ]:
# 13. Trening architektur
models={}; histories={}; rows=[]
for kind in MODELS_TO_RUN:
    print('\n'+'='*70+'\nMODEL:',kind)
    m=SentimentRNN(len(itos),kind,EMBED_DIM,HIDDEN_DIM,NUM_LAYERS,DROPOUT,BIDIRECTIONAL).to(DEVICE)
    params=sum(p.numel() for p in m.parameters() if p.requires_grad); print('Parametry:',f'{params:,}')
    h=train_model(m,train_loader,val_loader,EPOCHS,LEARNING_RATE); vl,va=evaluate(m,val_loader,nn.BCEWithLogitsLoss())
    models[kind]=m; histories[kind]=h; rows.append({'model':kind,'parameters':params,'val_loss':vl,'val_accuracy':va})
results=pd.DataFrame(rows).sort_values('val_loss').reset_index(drop=True)
results

In [ ]:
# 14. Krzywe walidacyjne
fig,ax=plt.subplots(figsize=(8,5))
for k,h in histories.items(): ax.plot(range(1,len(h['val_loss'])+1),h['val_loss'],marker='o',label=k)
ax.set_title('Validation loss'); ax.set_xlabel('epoka'); ax.legend(); plt.show()
fig,ax=plt.subplots(figsize=(8,5))
for k,h in histories.items(): ax.plot(range(1,len(h['val_acc'])+1),h['val_acc'],marker='o',label=k)
ax.set_title('Validation accuracy'); ax.set_xlabel('epoka'); ax.set_ylim(0,1); ax.legend(); plt.show()

## 8. Test najlepszego modelu
Model wybieramy po validation loss. Zbiór testowy służy dopiero do finalnej oceny.

In [ ]:
# 15. Finalny test
best_kind=results.iloc[0].model; best_model=models[best_kind]; criterion=nn.BCEWithLogitsLoss()
test_loss,test_acc=evaluate(best_model,test_loader,criterion)
all_preds=[]; all_labels=[]; all_probs=[]; best_model.eval()
with torch.no_grad():
    for x,L,y in test_loader:
        p=torch.sigmoid(best_model(x.to(DEVICE),L)); pred=(p>=.5).long()
        all_probs.extend(p.cpu().tolist()); all_preds.extend(pred.cpu().tolist()); all_labels.extend(y.long().tolist())
print('Najlepszy:',best_kind); print(f'Test loss {test_loss:.4f} | accuracy {test_acc:.4f}')
print(classification_report(all_labels,all_preds,target_names=['negative','positive'],digits=4))

In [ ]:
# 16. Confusion matrix
cm=confusion_matrix(all_labels,all_preds)
ConfusionMatrixDisplay(cm,display_labels=['negative','positive']).plot(values_format='d')
plt.title('Confusion matrix — '+best_kind); plt.show()

## 9. Predykcja własnych zdań

In [ ]:
# 17. Predykcja
def predict_sentiment(text,model=best_model):
    ids,L=encode(clean_text(text)); x=torch.tensor([ids],dtype=torch.long,device=DEVICE); lens=torch.tensor([L])
    model.eval()
    with torch.no_grad(): p=torch.sigmoid(model(x,lens)).item()
    return {'text':text,'label':'positive' if p>=.5 else 'negative','p_positive':round(p,4),'p_negative':round(1-p,4)}
examples=['I love this product! It works perfectly.','This update is awful and completely broken.','Not bad at all, I actually like it.','Great, another crash... exactly what I needed.','@support the new camera is fantastic! #happy']
pd.DataFrame([predict_sentiment(t) for t in examples])

## 10. Stress test: negacja, ironia, sentyment mieszany
To ręczny test OOD, nie część formalnego testu statystycznego.

In [ ]:
# 18. Stress test
hard=pd.DataFrame([(0,'I do not love this app at all.'),(1,'This is not bad, actually it works quite well.'),(0,'Great, it crashed again. Exactly what I needed.'),(1,'I expected it to be terrible, but it is fantastic.'),(0,'The design is beautiful but the service is awful.'),(1,'The delivery was slow, yet the product is excellent.'),(0,'I used to love this app before the broken update.'),(1,'I hate waiting, but the support team solved everything.')],columns=['expected','text'])
pred=pd.DataFrame([predict_sentiment(t) for t in hard.text]); res=pd.concat([hard,pred[['label','p_positive']]],axis=1)
res['predicted']=(res.label=='positive').astype(int); res['correct']=res.predicted==res.expected
print('Stress-test accuracy:',round(res.correct.mean(),3)); res

In [ ]:
# 19. Zapis modelu
MODEL_FILE=f'best_{best_kind.lower()}_sentiment140_{DATASET_SIZE}.pt'
torch.save({'model_type':best_kind,'model_state_dict':best_model.state_dict(),'stoi':stoi,'itos':itos,'max_len':MAX_LEN,'embed_dim':EMBED_DIM,'hidden_dim':HIDDEN_DIM,'num_layers':NUM_LAYERS,'dropout':DROPOUT,'bidirectional':BIDIRECTIONAL,'dataset_size':DATASET_SIZE},MODEL_FILE)
print('Zapisano:',MODEL_FILE)

## 11. Zadanie dla kursantów: 20k → 200k → 1,6M

Uruchom notebook trzy razy, zmieniając tylko `DATASET_SIZE`, i porównaj accuracy, F1, validation loss, czas treningu oraz błędy na stress-teście.

**Do optymalizacji:** `MAX_VOCAB`, `MAX_LEN`, `EMBED_DIM`, `HIDDEN_DIM`, `NUM_LAYERS`, `BIDIRECTIONAL`, `DROPOUT`, `LEARNING_RATE`, `BATCH_SIZE`.

**Pytania głębsze:**
1. Czy 1,6 mln automatycznie oznaczonych przykładów jest lepsze niż mniejszy ręcznie opisany zbiór?
2. Czy sposób tworzenia etykiet Sentiment140 przez emotikony może wprowadzać specyficzne biasy?
3. Gdzie zwykły RNN przegrywa z LSTM/GRU?
4. Czy bidirectional pomaga?
5. Jak modele reagują na ironię, negację i sentyment mieszany?
6. Jak zmieni się wynik po zastąpieniu RNN małym transformerem, np. DistilBERT?